09x promotion x repurchase 2x2 EDA

Descriptive EDA only. No modeling, no train/test split, no prediction, no SHAP, no Optuna, no segmentation, no causal claim, and no feature selection decision.



In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import math
import shutil
import subprocess
import zipfile

import numpy as np
import pandas as pd

STEP = '09x_promotion_repurchase_2x2_EDA_260516'
START_TS = datetime.now()
warnings = []
errors = []
log_lines = []

def log(message):
    line = f'[{datetime.now().isoformat(timespec="seconds")}] {message}'
    log_lines.append(line)
    print(line)

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / '.git').exists() and (p / 'park.ingyeom').exists():
            return p.resolve()
    out = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
    root = Path(out).resolve()
    if not (root / 'park.ingyeom').exists():
        raise RuntimeError('repo root mismatch')
    return root

ROOT = find_repo_root()
PARK = ROOT / 'park.ingyeom'
NB_DIR = PARK / 'notebook' / STEP
NB_PATH = NB_DIR / f'{STEP}.ipynb'
PAYLOAD_PATH = NB_DIR / f'{STEP}_payload.txt'
OUT_DIR = PARK / 'reports' / 'audits' / STEP
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'
NOTE_PATH = PARK / 'note.md'
INPUT_06X = PARK / 'reports' / 'audits' / '06x_dataset_generation_260515'
INPUT_07X = PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515'
INPUT_08X = PARK / 'reports' / 'audits' / '08x_promotion_nonpromotion_EDA_260516'
DATA_DIR = PARK / 'data'

for d in [NB_DIR, OUT_DIR, ZIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

existing_payload = [p for p in OUT_DIR.iterdir() if p.name != '.ipynb_checkpoints' and not p.name.startswith('run_')]
if existing_payload:
    archive_dir = OUT_DIR / ('run_' + START_TS.strftime('%Y%m%d_%H%M%S'))
    archive_dir.mkdir(parents=True, exist_ok=True)
    for p in existing_payload:
        shutil.move(str(p), str(archive_dir / p.name))
    log(f'preserved existing output payload under {archive_dir}')

if ZIP_PATH.exists():
    zip_archive_dir = ZIP_DIR / ('run_' + START_TS.strftime('%Y%m%d_%H%M%S'))
    zip_archive_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(str(ZIP_PATH), str(zip_archive_dir / ZIP_PATH.name))
    log(f'preserved existing review zip under {zip_archive_dir}')

def inside_park(path):
    try:
        Path(path).resolve().relative_to(PARK.resolve())
        return True
    except Exception:
        return False

def write_csv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def norm_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in {'true', 'yes', '1', 'y'}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def fingerprint_one(path, role):
    if not path.exists():
        return {'file_path': str(path), 'file_role': role, 'sha256': '', 'mtime': '', 'size': '', 'status': 'missing'}
    stat = path.stat()
    return {
        'file_path': str(path),
        'file_role': role,
        'sha256': sha256_file(path),
        'mtime': datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds'),
        'size': int(stat.st_size),
        'status': 'present',
    }

raw_source_names = [
    '(광일)Membership_v2_with_derived_features.csv',
    'Membership_v2.csv',
    'View_History_v2.csv',
    'User_Mapping_v2.csv',
    'Movie_Master_v2.csv',
    'Membership_train.csv',
    '변수_합집합_비교_v3.csv',
]
source_before = {name: fingerprint_one(DATA_DIR / name, 'raw_source_csv') for name in raw_source_names}
log(f'ROOT = {ROOT}')
log(f'PARK = {PARK}')
log(f'OUT_DIR = {OUT_DIR}')
log('source fingerprint before captured')



[2026-05-16T01:35:21] preserved existing output payload under C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\09x_promotion_repurchase_2x2_EDA_260516\run_20260516_013521
[2026-05-16T01:35:21] preserved existing review zip under C:\Code\ott-churn-prediction\park.ingyeom\zip\run_20260516_013521
[2026-05-16T01:35:21] ROOT = C:\Code\ott-churn-prediction
[2026-05-16T01:35:21] PARK = C:\Code\ott-churn-prediction\park.ingyeom
[2026-05-16T01:35:21] OUT_DIR = C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\09x_promotion_repurchase_2x2_EDA_260516
[2026-05-16T01:35:21] source fingerprint before captured


In [2]:
preflight_rows = []

def add_preflight(check_name, status, detail=''):
    preflight_rows.append({'check_name': check_name, 'status': status, 'detail': detail})

required_06x = [
    '06x_conservative_dataset.csv', '06x_expanded_dataset.csv', '06x_model_feature_lists.csv',
    '06x_dataset_schema_conservative.csv', '06x_dataset_schema_expanded.csv',
    '06x_scope_feature_policy.csv', '06x_caveat_register.csv', '06x_final_checks.csv'
]
required_07x = [
    '07x_feature_mapping_master.csv', '07x_AARRR_summary_by_feature_set.csv',
    '07x_conservative_AARRR_mapping.csv', '07x_expanded_AARRR_mapping.csv',
    '07x_scope_policy_handoff.csv', '07x_caveat_handoff.csv', '07x_downstream_EDA_handoff.csv',
    '07x_final_checks.csv'
]
required_08x = [
    '08x_dataset_scope_summary.csv', '08x_promotion_target_summary.csv',
    '08x_numeric_feature_group_comparison.csv', '08x_binary_feature_group_comparison.csv',
    '08x_feature_family_summary.csv', '08x_AARRR_stage_summary.csv',
    '08x_top_observed_differences_for_review.csv', '08x_caveat_and_claim_guardrail.csv',
    '08x_downstream_handoff.csv', '08x_redundancy_audit_handoff.csv', '08x_final_checks.csv'
]

add_preflight('06x folder exists', 'PASS' if INPUT_06X.exists() else 'FAIL', str(INPUT_06X))
add_preflight('07x folder exists', 'PASS' if INPUT_07X.exists() else 'FAIL', str(INPUT_07X))
add_preflight('08x folder exists', 'PASS' if INPUT_08X.exists() else 'FAIL', str(INPUT_08X))
for name in required_06x:
    add_preflight('required 06x files exist', 'PASS' if (INPUT_06X / name).exists() else 'FAIL', name)
for name in required_07x:
    add_preflight('required 07x files exist', 'PASS' if (INPUT_07X / name).exists() else 'FAIL', name)
for name in required_08x:
    add_preflight('required 08x files exist', 'PASS' if (INPUT_08X / name).exists() else 'FAIL', name)

def final_checks_pass(path):
    df = pd.read_csv(path)
    if 'status' not in df.columns:
        return False, 'status column missing'
    bad = df[~df['status'].astype(str).str.upper().isin(['PASS'])]
    return len(bad) == 0, f'{len(df)} checks, {len(bad)} non-PASS'

ok_06x, detail_06x = final_checks_pass(INPUT_06X / '06x_final_checks.csv')
ok_07x, detail_07x = final_checks_pass(INPUT_07X / '07x_final_checks.csv')
ok_08x, detail_08x = final_checks_pass(INPUT_08X / '08x_final_checks.csv')
add_preflight('06x final checks pass', 'PASS' if ok_06x else 'FAIL', detail_06x)
add_preflight('07x final checks pass', 'PASS' if ok_07x else 'FAIL', detail_07x)
add_preflight('08x final checks pass', 'PASS' if ok_08x else 'FAIL', detail_08x)

conservative = pd.read_csv(INPUT_06X / '06x_conservative_dataset.csv')
expanded = pd.read_csv(INPUT_06X / '06x_expanded_dataset.csv')
feature_lists = pd.read_csv(INPUT_06X / '06x_model_feature_lists.csv')
mapping = pd.read_csv(INPUT_07X / '07x_feature_mapping_master.csv')
target_08x = pd.read_csv(INPUT_08X / '08x_promotion_target_summary.csv')

add_preflight('conservative dataset loaded', 'PASS', str(conservative.shape))
add_preflight('expanded dataset loaded', 'PASS', str(expanded.shape))
add_preflight('is_promotion split available', 'PASS' if 'is_promotion' in expanded.columns else 'FAIL', 'expanded dataset')
target_available = 'is_repurchase' in conservative.columns and 'is_repurchase' in expanded.columns
add_preflight('is_repurchase target available', 'PASS' if target_available else 'FAIL', 'conservative and expanded')
add_preflight('source fingerprint before captured', 'PASS', f'{len(source_before)} files')
add_preflight('output folder created', 'PASS' if OUT_DIR.exists() else 'FAIL', str(OUT_DIR))

align_ok = (
    len(conservative) == len(expanded)
    and conservative['USER_KEY'].equals(expanded['USER_KEY'])
    and conservative['is_repurchase'].equals(expanded['is_repurchase'])
)
add_preflight('conservative expanded row alignment verified', 'PASS' if align_ok else 'FAIL', 'USER_KEY and is_repurchase exact row order')
if not align_ok:
    add_preflight('stop_reason', 'FAIL', 'conservative_expanded_row_alignment_failed')
    write_csv(pd.DataFrame(preflight_rows), '09x_preflight_input_validation.csv')
    raise RuntimeError('Conservative and expanded row alignment failed')

cohort_map = {
    (1, 1): 'promotion_repurchase',
    (1, 0): 'promotion_nonrepurchase',
    (0, 1): 'nonpromotion_repurchase',
    (0, 0): 'nonpromotion_nonrepurchase',
}

def add_cohort_label(df):
    out = df.copy()
    out['is_promotion'] = pd.to_numeric(out['is_promotion'], errors='coerce').astype('Int64')
    out['is_repurchase'] = pd.to_numeric(out['is_repurchase'], errors='coerce').astype('Int64')
    out['cohort_2x2_label'] = [
        cohort_map.get((int(p), int(r)), 'invalid_or_missing') if pd.notna(p) and pd.notna(r) else 'invalid_or_missing'
        for p, r in zip(out['is_promotion'], out['is_repurchase'])
    ]
    return out

conservative_analysis = conservative.copy()
conservative_analysis['is_promotion'] = expanded['is_promotion']
conservative_analysis = add_cohort_label(conservative_analysis)
expanded_analysis = add_cohort_label(expanded)
add_preflight('2x2 cohort labels generated for analysis only', 'PASS', 'role=analysis_group_label; use_as_feature=False')
add_preflight('stop_reason', 'PASS', 'none')
write_csv(pd.DataFrame(preflight_rows), '09x_preflight_input_validation.csv')

analysis_group_label_register = pd.DataFrame([{
    'safe_model_feature_name': 'cohort_2x2_label',
    'role': 'analysis_group_label',
    'use_as_feature': False,
    'saved_to_model_input': False,
    'notes': 'Temporary notebook grouping label for 09x EDA only.'
}])
log('inputs loaded and 2x2 analysis labels generated')



[2026-05-16T01:35:21] inputs loaded and 2x2 analysis labels generated


In [3]:
datasets = {
    'conservative_safe_22': conservative_analysis,
    'expanded_feature_set': expanded_analysis,
}
mapping_use = mapping[mapping['use_as_feature'].map(norm_bool)].copy()
exclude_from_profiles = {'USER_KEY', 'is_repurchase', 'is_promotion', 'cohort_2x2_label'}
feature_meta = {}
for _, row in mapping_use.iterrows():
    fs = row['feature_set_name']
    feat = row['safe_model_feature_name']
    if feat not in exclude_from_profiles:
        feature_meta[(fs, feat)] = row.to_dict()

cohort_rows = []
for fs, df in datasets.items():
    total = len(df)
    promo_counts = df.groupby('is_promotion').size().to_dict()
    rep_counts = df.groupby('is_repurchase').size().to_dict()
    for label, group in df.groupby('cohort_2x2_label', dropna=False):
        p = int(group['is_promotion'].iloc[0]) if len(group) else np.nan
        r = int(group['is_repurchase'].iloc[0]) if len(group) else np.nan
        cohort_rows.append({
            'feature_set_name': fs,
            'cohort_2x2_label': label,
            'is_promotion_value': p,
            'is_repurchase_value': r,
            'row_count': int(len(group)),
            'row_share_within_feature_set': len(group) / total if total else np.nan,
            'row_share_within_promotion_group': len(group) / promo_counts.get(p, np.nan),
            'row_share_within_repurchase_group': len(group) / rep_counts.get(r, np.nan),
            'notes': '2x2 cohort label is analysis group only, not a model feature.'
        })
cohort_summary = pd.DataFrame(cohort_rows).sort_values(['feature_set_name', 'is_promotion_value', 'is_repurchase_value'], ascending=[True, False, False])
write_csv(cohort_summary, '09x_2x2_cohort_summary.csv')

target_rows = []
for fs, df in datasets.items():
    group_stats = {}
    for pval, g in df.groupby('is_promotion'):
        name = 'promotion' if int(pval) == 1 else 'nonpromotion'
        rep1 = int((g['is_repurchase'] == 1).sum())
        rep0 = int((g['is_repurchase'] == 0).sum())
        rate = rep1 / len(g) if len(g) else np.nan
        group_stats[name] = {'row_count': len(g), 'rep1': rep1, 'rep0': rep0, 'rate': rate}
    for name, stats in group_stats.items():
        other = 'nonpromotion' if name == 'promotion' else 'promotion'
        other_rate = group_stats.get(other, {}).get('rate', np.nan)
        target_rows.append({
            'feature_set_name': fs,
            'promotion_group': name,
            'row_count': int(stats['row_count']),
            'repurchase_1_count': int(stats['rep1']),
            'repurchase_0_count': int(stats['rep0']),
            'repurchase_rate': stats['rate'],
            'nonrepurchase_rate': 1 - stats['rate'] if pd.notna(stats['rate']) else np.nan,
            'rate_difference_vs_other_promotion_group': stats['rate'] - other_rate if pd.notna(other_rate) else np.nan,
            'safe_interpretation': f'{name} group has an observed repurchase rate in 09x 2x2 EDA.',
            'forbidden_interpretation': 'promotion caused churn, prevented churn, or changed repurchase behavior.'
        })
target_summary = pd.DataFrame(target_rows).sort_values(['feature_set_name', 'promotion_group'])

compare_cols = ['feature_set_name', 'promotion_group', 'row_count', 'repurchase_1_count', 'repurchase_0_count']
left = target_summary[compare_cols].sort_values(compare_cols).reset_index(drop=True)
right = target_08x.rename(columns={'group_name': 'promotion_group'})[compare_cols].sort_values(compare_cols).reset_index(drop=True)
target_matches_08x = left.equals(right)
if not target_matches_08x:
    warnings.append('09x target summary does not match 08x target summary')
write_csv(target_summary, '09x_2x2_target_rate_summary.csv')
log(f'cohort and target summaries created; target_matches_08x={target_matches_08x}')



[2026-05-16T01:35:21] cohort and target summaries created; target_matches_08x=True


In [4]:
def is_binary_series(s):
    vals = pd.Series(s).dropna().unique()
    return len(vals) > 0 and len(vals) <= 2 and set(vals).issubset({0, 1, 0.0, 1.0})

def smd(a, b):
    a = pd.to_numeric(a, errors='coerce').dropna()
    b = pd.to_numeric(b, errors='coerce').dropna()
    if len(a) == 0 or len(b) == 0:
        return np.nan
    va = a.var(ddof=1) if len(a) > 1 else 0
    vb = b.var(ddof=1) if len(b) > 1 else 0
    pooled = math.sqrt((va + vb) / 2) if (va + vb) > 0 else 0
    if pooled == 0:
        return 0.0 if abs(a.mean() - b.mean()) == 0 else np.nan
    return (a.mean() - b.mean()) / pooled

def meta_for(fs, feat):
    return feature_meta.get((fs, feat), {
        'AARRR_stage': '', 'feature_family': '', 'timing_family': '',
        'caveat_flag': False, 'caveat_reason': '', 'needs_user_review': 0
    })

def review_int(x):
    try:
        if str(x).strip() == '':
            return 0
        return int(float(x))
    except Exception:
        return 0

def base_meta(fs, feat):
    m = meta_for(fs, feat)
    return {
        'feature_set_name': fs,
        'safe_model_feature_name': feat,
        'AARRR_stage': m.get('AARRR_stage', ''),
        'feature_family': m.get('feature_family', ''),
        'timing_family': m.get('timing_family', ''),
        'caveat_flag': bool(norm_bool(m.get('caveat_flag', False))),
        'caveat_reason': m.get('caveat_reason', ''),
        'needs_user_review': review_int(m.get('needs_user_review', 0)),
        'interpretation_guardrail': 'Observed 2x2 cohort difference only. Not causal, not feature importance, and not feature selection.'
    }

numeric_rows = []
binary_rows = []
feature_kind = {}
for fs, df in datasets.items():
    feats = [feat for (mfs, feat) in feature_meta if mfs == fs and feat in df.columns and feat not in exclude_from_profiles]
    for feat in feats:
        s = pd.to_numeric(df[feat], errors='coerce')
        kind = 'binary' if is_binary_series(s) else 'numeric'
        feature_kind[(fs, feat)] = kind
        for label, group in df.groupby('cohort_2x2_label', dropna=False):
            g = pd.to_numeric(group[feat], errors='coerce')
            common = base_meta(fs, feat)
            if kind == 'binary':
                binary_rows.append({
                    **common,
                    'cohort_2x2_label': label,
                    'n': int(g.notna().sum()),
                    'positive_count': int((g == 1).sum()),
                    'positive_rate': g.mean() if g.notna().sum() else np.nan,
                    'missing_count': int(g.isna().sum()),
                })
            else:
                numeric_rows.append({
                    **common,
                    'cohort_2x2_label': label,
                    'n': int(g.notna().sum()),
                    'mean': g.mean(),
                    'median': g.median(),
                    'std': g.std(ddof=1),
                    'min': g.min(),
                    'q25': g.quantile(0.25),
                    'q75': g.quantile(0.75),
                    'max': g.max(),
                    'missing_count': int(g.isna().sum()),
                })

numeric_profile = pd.DataFrame(numeric_rows)
binary_profile = pd.DataFrame(binary_rows)
write_csv(numeric_profile, '09x_numeric_2x2_feature_profile.csv')
write_csv(binary_profile, '09x_binary_2x2_feature_profile.csv')
log(f'numeric profile rows={len(numeric_profile)}, binary profile rows={len(binary_profile)}')



[2026-05-16T01:35:23] numeric profile rows=276, binary profile rows=128


In [5]:
comparison_pairs = {
    'within_promotion_repurchase': ('promotion_repurchase', 'promotion_nonrepurchase', 'repurchase_value', 'nonrepurchase_value', 'difference_repurchase_minus_nonrepurchase'),
    'within_nonpromotion_repurchase': ('nonpromotion_repurchase', 'nonpromotion_nonrepurchase', 'repurchase_value', 'nonrepurchase_value', 'difference_repurchase_minus_nonrepurchase'),
    'between_promotion_status_within_repurchase_1': ('promotion_repurchase', 'nonpromotion_repurchase', 'promotion_value', 'nonpromotion_value', 'difference_promotion_minus_nonpromotion'),
    'between_promotion_status_within_repurchase_0': ('promotion_nonrepurchase', 'nonpromotion_nonrepurchase', 'promotion_value', 'nonpromotion_value', 'difference_promotion_minus_nonpromotion'),
}

def feature_value(df, feat, label):
    s = pd.to_numeric(df.loc[df['cohort_2x2_label'] == label, feat], errors='coerce')
    return s.mean() if s.notna().sum() else np.nan

def comparison_rows_for(scope):
    label_a, label_b, col_a, col_b, diff_col = comparison_pairs[scope]
    rows = []
    for fs, df in datasets.items():
        feats = [feat for (mfs, feat) in feature_meta if mfs == fs and feat in df.columns and feat not in exclude_from_profiles]
        for feat in feats:
            kind = feature_kind.get((fs, feat), 'numeric')
            a = feature_value(df, feat, label_a)
            b = feature_value(df, feat, label_b)
            diff = a - b if pd.notna(a) and pd.notna(b) else np.nan
            if kind == 'binary':
                eff = abs(diff) if pd.notna(diff) else np.nan
                metric = 'absolute_rate_difference'
            else:
                sa = pd.to_numeric(df.loc[df['cohort_2x2_label'] == label_a, feat], errors='coerce')
                sb = pd.to_numeric(df.loc[df['cohort_2x2_label'] == label_b, feat], errors='coerce')
                eff = abs(smd(sa, sb))
                metric = 'absolute_standardized_mean_difference'
            rows.append({
                **base_meta(fs, feat),
                'comparison_scope': scope,
                'comparison_name': f'{label_a} vs {label_b}',
                'metric_type': kind,
                col_a: a,
                col_b: b,
                diff_col: diff,
                'abs_difference_or_abs_smd': eff,
                'effect_size_metric': metric,
                'not_a_feature_selection_decision': True,
            })
    return rows

all_comparison_rows = []
for scope in comparison_pairs:
    all_comparison_rows.extend(comparison_rows_for(scope))
all_comparisons = pd.DataFrame(all_comparison_rows)
within_promo = all_comparisons[all_comparisons['comparison_scope'] == 'within_promotion_repurchase'].drop(columns=['comparison_scope', 'comparison_name', 'promotion_value', 'nonpromotion_value', 'difference_promotion_minus_nonpromotion'], errors='ignore')
within_nonpromo = all_comparisons[all_comparisons['comparison_scope'] == 'within_nonpromotion_repurchase'].drop(columns=['comparison_scope', 'comparison_name', 'promotion_value', 'nonpromotion_value', 'difference_promotion_minus_nonpromotion'], errors='ignore')
between = all_comparisons[all_comparisons['comparison_scope'].str.startswith('between_')].drop(columns=['repurchase_value', 'nonrepurchase_value', 'difference_repurchase_minus_nonrepurchase'], errors='ignore')
write_csv(within_promo, '09x_within_promotion_repurchase_comparison.csv')
write_csv(within_nonpromo, '09x_within_nonpromotion_repurchase_comparison.csv')
write_csv(between, '09x_between_promotion_status_within_repurchase_comparison.csv')
log(f'comparison rows created={len(all_comparisons)}')



[2026-05-16T01:35:24] comparison rows created=404


In [6]:
family_rows = []
stage_rows = []
for fs in datasets:
    fs_features = mapping_use[(mapping_use['feature_set_name'] == fs) & (~mapping_use['safe_model_feature_name'].isin(exclude_from_profiles))]
    for scope in comparison_pairs:
        fs_scope = all_comparisons[(all_comparisons['feature_set_name'] == fs) & (all_comparisons['comparison_scope'] == scope)]
        for (stage, family), group in fs_features.groupby(['AARRR_stage', 'feature_family'], dropna=False):
            gm = fs_scope[(fs_scope['AARRR_stage'] == stage) & (fs_scope['feature_family'] == family)]
            vals = gm['abs_difference_or_abs_smd'].dropna()
            top = gm.sort_values('abs_difference_or_abs_smd', ascending=False)['safe_model_feature_name'].head(5).tolist() if len(gm) else []
            family_rows.append({
                'feature_set_name': fs,
                'comparison_scope': scope,
                'AARRR_stage': stage,
                'feature_family': family,
                'feature_count': int(group['safe_model_feature_name'].nunique()),
                'compared_feature_count': int(gm['safe_model_feature_name'].nunique()),
                'avg_abs_difference_or_smd': vals.mean() if len(vals) else np.nan,
                'max_abs_difference_or_smd': vals.max() if len(vals) else np.nan,
                'top_diff_features': ', '.join(top),
                'caveat_count': int(gm['caveat_flag'].sum()) if len(gm) else 0,
                'needs_user_review_count': int(gm['needs_user_review'].sum()) if len(gm) else 0,
                'summary_note': 'Family summary supports descriptive review only. It is not a feature removal or importance decision.'
            })
        for stage, group in fs_features.groupby('AARRR_stage', dropna=False):
            gm = fs_scope[fs_scope['AARRR_stage'] == stage]
            vals = gm['abs_difference_or_abs_smd'].dropna()
            top = gm.sort_values('abs_difference_or_abs_smd', ascending=False)['safe_model_feature_name'].head(5).tolist() if len(gm) else []
            stage_rows.append({
                'feature_set_name': fs,
                'comparison_scope': scope,
                'AARRR_stage': stage,
                'feature_count': int(group['safe_model_feature_name'].nunique()),
                'compared_feature_count': int(gm['safe_model_feature_name'].nunique()),
                'avg_abs_difference_or_smd': vals.mean() if len(vals) else np.nan,
                'max_abs_difference_or_smd': vals.max() if len(vals) else np.nan,
                'top_diff_features': ', '.join(top),
                'interpretation_guardrail': 'Observed 2x2 difference only. No causal, importance, or selection claim.',
                'notes': 'AARRR stage summary is descriptive only.'
            })
        if 'Referral' not in set(fs_features['AARRR_stage'].dropna().astype(str)):
            stage_rows.append({
                'feature_set_name': fs,
                'comparison_scope': scope,
                'AARRR_stage': 'Referral',
                'feature_count': 0,
                'compared_feature_count': 0,
                'avg_abs_difference_or_smd': np.nan,
                'max_abs_difference_or_smd': np.nan,
                'top_diff_features': '',
                'interpretation_guardrail': 'Referral has no directly observed feature in this dataset.',
                'notes': 'Do not claim Referral was validated by data in 09x.'
            })

feature_family_summary = pd.DataFrame(family_rows)
arr_stage_summary = pd.DataFrame(stage_rows)
write_csv(feature_family_summary, '09x_feature_family_2x2_summary.csv')
write_csv(arr_stage_summary, '09x_AARRR_stage_2x2_summary.csv')

top_rows = []
for (fs, scope), group in all_comparisons.groupby(['feature_set_name', 'comparison_scope']):
    for rank, (_, row) in enumerate(group.sort_values('abs_difference_or_abs_smd', ascending=False).head(20).iterrows(), 1):
        group_a = comparison_pairs[scope][0]
        group_b = comparison_pairs[scope][1]
        if scope.startswith('within_'):
            group_a_value = row.get('repurchase_value', np.nan)
            group_b_value = row.get('nonrepurchase_value', np.nan)
            difference = row.get('difference_repurchase_minus_nonrepurchase', np.nan)
        else:
            group_a_value = row.get('promotion_value', np.nan)
            group_b_value = row.get('nonpromotion_value', np.nan)
            difference = row.get('difference_promotion_minus_nonpromotion', np.nan)
        top_rows.append({
            'feature_set_name': fs,
            'comparison_scope': scope,
            'rank': rank,
            'safe_model_feature_name': row['safe_model_feature_name'],
            'metric_type': row['metric_type'],
            'AARRR_stage': row['AARRR_stage'],
            'feature_family': row['feature_family'],
            'timing_family': row['timing_family'],
            'effect_size_or_abs_smd': row['abs_difference_or_abs_smd'],
            'group_a': group_a,
            'group_a_value': group_a_value,
            'group_b': group_b,
            'group_b_value': group_b_value,
            'difference': difference,
            'caveat_flag': row['caveat_flag'],
            'caveat_reason': row['caveat_reason'],
            'needs_user_review': row['needs_user_review'],
            'recommended_next_step': 'Review distribution in 10x and audit redundancy or leakage suspect risk in 10x or 11x before any modeling use.',
            'not_a_feature_selection_decision': True,
        })
top_diffs = pd.DataFrame(top_rows)
write_csv(top_diffs, '09x_top_2x2_observed_differences_for_review.csv')
log('family, AARRR, and top difference summaries created')



[2026-05-16T01:35:25] family, AARRR, and top difference summaries created


In [7]:
context_features = ['is_user_verified', 'payment_is_ios', 'age_group', 'payment_is_mobile', 'is_female', 'is_premium', 'is_male', 'payment_is_pc']
context_rows = []
for feat in context_features:
    for fs, df in datasets.items():
        present = feat in df.columns and (fs, feat) in feature_meta
        values = {}
        near_constant = False
        if present:
            for label, group in df.groupby('cohort_2x2_label'):
                s = pd.to_numeric(group[feat], errors='coerce')
                values[label] = float(s.mean()) if s.notna().sum() else np.nan
                vc = s.dropna().value_counts(normalize=True)
                if len(vc) > 0 and vc.iloc[0] >= 0.95:
                    near_constant = True
        m = meta_for(fs, feat) if present else {}
        group_proxy = present and (near_constant or feat in ['is_user_verified', 'payment_is_ios', 'payment_is_mobile', 'payment_is_pc', 'is_female', 'is_male'])
        context_rows.append({
            'safe_model_feature_name': feat,
            'present_in_feature_set': bool(present),
            'feature_set_name': fs,
            'AARRR_stage': m.get('AARRR_stage', ''),
            'feature_family': m.get('feature_family', ''),
            'cohort_2x2_values': '; '.join([f'{k}={v}' for k, v in values.items()]),
            'near_constant_in_any_cohort': bool(near_constant),
            'group_proxy_risk': bool(group_proxy),
            'redundancy_or_leakage_review_needed': bool(present),
            'user_approval_required': True,
            'recommended_next_step': 'Pass to 10x or 11x redundancy, near-constant, group proxy, and leakage suspect audit. Do not remove here.',
            'notes': 'Observed 08x difference is not cause, importance, or final feature usage approval.'
        })
context_review = pd.DataFrame(context_rows)
write_csv(context_review, '09x_context_profile_proxy_risk_review.csv')

usage_patterns = ['cold_start', 'watch_time', 'watch_session', 'retention', 'diff_between', 'recency', 'inactive', 'gap', 'is_only_w', 'watch_days', 'active_ratio']
usage_candidates = all_comparisons[all_comparisons['safe_model_feature_name'].str.contains('|'.join(usage_patterns), case=False, regex=True, na=False)].copy()
usage_rows = []
for (fs, feat, scope), group in usage_candidates.groupby(['feature_set_name', 'safe_model_feature_name', 'comparison_scope']):
    row = group.sort_values('abs_difference_or_abs_smd', ascending=False).iloc[0]
    usage_rows.append({
        'safe_model_feature_name': feat,
        'feature_set_name': fs,
        'AARRR_stage': row['AARRR_stage'],
        'feature_family': row['feature_family'],
        'comparison_scope': scope,
        'observed_pattern': 'Repurchase and nonrepurchase or promotion-status cohorts show an observed descriptive difference in this comparison scope.',
        'effect_size_or_abs_smd': row['abs_difference_or_abs_smd'],
        'caveat_flag': row['caveat_flag'],
        'needs_user_review': row['needs_user_review'],
        'recommended_next_step': 'Recheck feature distributions in 10x and avoid causal wording.',
        'notes': 'Do not say this feature causes repurchase.'
    })
usage_review = pd.DataFrame(usage_rows).sort_values(['feature_set_name', 'comparison_scope', 'effect_size_or_abs_smd'], ascending=[True, True, False])
write_csv(usage_review, '09x_usage_retention_2x2_review.csv')
log('context proxy and usage retention reviews created')



[2026-05-16T01:35:25] context proxy and usage retention reviews created


In [8]:
caveats = [
    ('observed_difference_only', 'observed difference only', '', '09x', '2x2 cohorts show observed descriptive differences.', 'The difference is causal or actionable by itself.', 'Use observed difference only wording.'),
    ('no_causal_claim', 'no causal claim', '', '09x', '09x does not estimate causal effects.', 'Promotion caused churn or repurchase.', 'Do not use causal verbs.'),
    ('no_uplift_claim', 'no uplift claim', '', '09x', '09x does not estimate uplift.', 'Promotion uplifted retention.', 'Use descriptive comparison only.'),
    ('no_marketing_effectiveness_claim', 'no marketing effectiveness claim', '', '09x', '09x does not evaluate marketing effectiveness.', 'Promotion was ineffective.', 'Use observed group difference only.'),
    ('no_feature_importance_claim', 'no feature importance claim', '', '09x', 'EDA differences are not model importance.', 'Top difference is an important feature.', 'Say review candidate.'),
    ('no_feature_selection_decision', 'no feature selection decision', '', '09x', 'No feature is selected or removed in 09x.', 'Use or remove this feature.', 'Require later audit and user approval.'),
    ('no_model_performance_claim', 'no model performance claim', '', '09x', 'No model was trained or evaluated.', 'This improves AUC.', 'No performance wording.'),
    ('shap_not_performed', 'SHAP not performed', '', '09x', 'SHAP was not performed.', 'SHAP confirms this feature.', 'Reserve SHAP wording for SHAP step.'),
    ('modeling_not_performed', 'modeling not performed', '', '09x', 'Modeling was not performed.', 'The model learned this pattern.', 'No model wording.'),
    ('segmentation_not_performed', 'segmentation not performed', '', '09x', 'Segmentation was not performed.', 'This is a final segment.', 'No segment claim.'),
    ('is_promotion_split_key', 'is_promotion split key caveat', 'is_promotion', '09x', 'is_promotion is a split key.', 'is_promotion is a model feature here.', 'Do not treat split key as feature.'),
    ('is_repurchase_target', 'is_repurchase target caveat', 'is_repurchase', '09x', 'is_repurchase is target.', 'is_repurchase is a feature.', 'Exclude target from profile features.'),
    ('cohort_label_analysis_only', '2x2 cohort label is analysis group only', 'cohort_2x2_label', '09x', 'cohort_2x2_label is an analysis group label only.', 'Use cohort label as model input.', 'role=analysis_group_label, use_as_feature=False.'),
    ('is_churn_prevented_context', 'is_churn_prevented historical context caveat', 'is_churn_prevented', '09x', 'Historical context field needs leakage review.', 'This proves churn was prevented.', 'Pass to leakage suspect audit if used later.'),
    ('cold_start_fixed', 'cold_start_fixed caveat', 'is_cold_start_3d_fixed,is_cold_start_7d_fixed', '09x', 'Cold-start fixed variables are descriptive onboarding indicators.', 'Cold start caused repurchase.', 'Use observed difference wording only.'),
    ('old_movie_ratio_5y_mismatch', 'old_movie_ratio_5y 9-row mismatch caveat', 'old_movie_ratio_5y', '09x', 'Known 9-row mismatch caveat remains inherited.', 'This field is fully formula-validated.', 'Carry caveat forward.'),
    ('genre_multi_category', 'genre multi-category caveat', 'genre ratios', '09x', 'Genre fields may reflect multi-category mapping caveats.', 'Genre ratio is exact causal preference.', 'Use descriptive caveat.'),
    ('under_threshold', 'under_1m/5m <= threshold caveat', 'watch_ratio_under_1m,watch_ratio_under_5m', '09x', 'Threshold feature definition needs careful wording.', 'Threshold features prove intent.', 'Use definition-level caveat.'),
    ('context_profile_proxy', 'context/profile/payment group proxy risk caveat', 'context/profile/payment features', '09x', 'Context, profile, and payment fields may act as group proxies.', 'These are approved final features.', 'Send to 10x or 11x redundancy, proxy, and leakage audit.'),
]
guardrail = pd.DataFrame(caveats, columns=['caveat_id', 'caveat_topic', 'applies_to_feature', 'applies_to_stage', 'safe_claim', 'unsafe_claim', 'required_wording'])
write_csv(guardrail, '09x_caveat_and_claim_guardrail.csv')

handoff_rows = []
for fs in datasets:
    fs_top = top_diffs[top_diffs['feature_set_name'] == fs].sort_values('effect_size_or_abs_smd', ascending=False)['safe_model_feature_name'].dropna().unique()[:20]
    fs_families = feature_family_summary[feature_family_summary['feature_set_name'] == fs]['feature_family'].dropna().unique()[:12]
    handoff_rows.extend([
        {'downstream_step': '10x_feature_distribution_EDA', 'handoff_topic': 'feature families to review', 'feature_set_name': fs, 'related_features': ', '.join(fs_families), 'reason': '09x summarized observed 2x2 differences by feature family.', 'required_action': 'Review distributions and missingness without feature removal.', 'risk_if_ignored': 'Descriptive distribution issues may be missed.', 'user_approval_required': False, 'notes': 'Family summary is not an importance ranking.'},
        {'downstream_step': '10x_feature_distribution_EDA', 'handoff_topic': '09x top difference feature distribution recheck', 'feature_set_name': fs, 'related_features': ', '.join(fs_top), 'reason': 'Largest observed 2x2 differences need distribution-level sanity check.', 'required_action': 'Reconfirm shape, missingness, outliers, and cohort-level stability.', 'risk_if_ignored': 'Top differences may be artifacts of skew or near constants.', 'user_approval_required': False, 'notes': 'Not a feature selection decision.'},
        {'downstream_step': '10x_or_11x', 'handoff_topic': 'multicollinearity and feature redundancy audit required', 'feature_set_name': fs, 'related_features': 'all use_as_feature fields', 'reason': '09x intentionally does not run redundancy audit.', 'required_action': 'Run VIF, pairwise correlation, redundancy cluster, near-constant, duplicate-like, and leakage-suspect audits.', 'risk_if_ignored': 'Expanded feature usage may be unstable or redundant.', 'user_approval_required': True, 'notes': 'No feature removal allowed without user approval.'},
        {'downstream_step': '11x_modeling_preflight', 'handoff_topic': 'conservative vs expanded feature usage confirmation', 'feature_set_name': fs, 'related_features': 'feature-set contract', 'reason': '09x kept conservative and expanded analyses separate.', 'required_action': 'Confirm actual feature list and scope before any model fit.', 'risk_if_ignored': 'Feature-set mixing or accidental is_promotion usage may occur.', 'user_approval_required': True, 'notes': '11x and 12x must re-check actual expanded feature usage.'},
    ])

handoff_rows.extend([
    {'downstream_step': '10x_or_11x', 'handoff_topic': 'VIF required', 'feature_set_name': 'both', 'related_features': 'numeric features', 'reason': 'Multicollinearity not tested in 09x.', 'required_action': 'Compute VIF where method assumptions are acceptable.', 'risk_if_ignored': 'Redundant predictors may distort interpretation.', 'user_approval_required': True, 'notes': 'Removal is not allowed here.'},
    {'downstream_step': '10x_or_11x', 'handoff_topic': 'pairwise correlation required', 'feature_set_name': 'both', 'related_features': 'numeric and binary features', 'reason': 'Pairwise redundancy not tested in 09x.', 'required_action': 'Create correlation pair table.', 'risk_if_ignored': 'Duplicate-like signals may be missed.', 'user_approval_required': True, 'notes': 'Use as audit evidence only.'},
    {'downstream_step': '10x_or_11x', 'handoff_topic': 'feature family redundancy cluster required', 'feature_set_name': 'both', 'related_features': 'all feature families', 'reason': 'Families contain related variables.', 'required_action': 'Cluster or group redundant features by family.', 'risk_if_ignored': 'Interpretation may double-count signals.', 'user_approval_required': True, 'notes': 'No automatic removal.'},
    {'downstream_step': '10x_or_11x', 'handoff_topic': 'near-constant feature audit required', 'feature_set_name': 'expanded_feature_set', 'related_features': 'is_user_verified, payment_is_ios and all low-variation fields', 'reason': '09x context review found proxy or near-constant risk candidates.', 'required_action': 'Audit dominant value share overall and by cohort.', 'risk_if_ignored': 'Group proxy or near-constant features may be overinterpreted.', 'user_approval_required': True, 'notes': 'Do not remove without approval.'},
    {'downstream_step': '10x_or_11x', 'handoff_topic': 'duplicate-like feature audit required', 'feature_set_name': 'both', 'related_features': 'all fields', 'reason': 'Duplicate-like features were not tested in 09x.', 'required_action': 'Run exact and near-exact equality checks.', 'risk_if_ignored': 'Redundant feature families may inflate interpretation.', 'user_approval_required': True, 'notes': 'Audit only.'},
    {'downstream_step': '11x_modeling_preflight', 'handoff_topic': 'target leakage suspect audit required', 'feature_set_name': 'expanded_feature_set', 'related_features': 'is_churn_prevented, context/profile/payment and policy-caveat fields', 'reason': 'Potential leakage or proxy variables require pre-model gate.', 'required_action': 'Review timing, source, and target relation before any model fit.', 'risk_if_ignored': 'Modeling may use invalid predictors.', 'user_approval_required': True, 'notes': '09x does not approve feature use.'},
    {'downstream_step': '10x_or_11x', 'handoff_topic': 'is_user_verified group proxy risk check', 'feature_set_name': 'expanded_feature_set', 'related_features': 'is_user_verified', 'reason': '08x and 09x indicate proxy-risk review candidate.', 'required_action': 'Check cohort distribution, redundancy, and leakage suspect status.', 'risk_if_ignored': 'Proxy-like interpretation may be overstated.', 'user_approval_required': True, 'notes': 'No removal decision in 09x.'},
    {'downstream_step': '10x_or_11x', 'handoff_topic': 'payment_is_ios near-constant and group proxy risk check', 'feature_set_name': 'expanded_feature_set', 'related_features': 'payment_is_ios', 'reason': 'User flagged near-constant/group-proxy risk.', 'required_action': 'Audit dominant share by cohort and redundancy with payment fields.', 'risk_if_ignored': 'Device/payment proxy may be overinterpreted.', 'user_approval_required': True, 'notes': 'No removal decision in 09x.'},
    {'downstream_step': '11x/12x', 'handoff_topic': 'actual expanded 80 feature usage recheck', 'feature_set_name': 'expanded_feature_set', 'related_features': 'expanded use_as_feature fields', 'reason': 'Feature count and use need pre-model verification.', 'required_action': 'Recount and validate expanded feature list before modeling.', 'risk_if_ignored': 'Expanded feature-set assumption may drift from actual contract.', 'user_approval_required': True, 'notes': '09x is not modeling approval.'},
    {'downstream_step': 'SHAP_step', 'handoff_topic': 'context/profile/payment interpretation overclaim guardrail', 'feature_set_name': 'expanded_feature_set', 'related_features': ', '.join(context_features), 'reason': 'Proxy-risk candidates require careful XAI wording.', 'required_action': 'Avoid causal or social-profile overinterpretation in SHAP.', 'risk_if_ignored': 'XAI narrative may become unsupported.', 'user_approval_required': True, 'notes': 'SHAP not performed in 09x.'},
])
write_csv(pd.DataFrame(handoff_rows), '09x_downstream_handoff.csv')

audit_items = [
    ('VIF', 'Detect multicollinearity among numeric features.', '10x or 11x', 'both', 'all numeric families', 'variance inflation factor', 'VIF table by feature'),
    ('pairwise correlation', 'Find highly correlated feature pairs.', '10x or 11x', 'both', 'all numeric and binary families', 'Pearson or Spearman correlation matrix and high-pair table', 'high-correlation pair table'),
    ('feature family redundancy cluster', 'Summarize overlapping signals within feature families.', '10x or 11x', 'both', 'usage, retention, genre, device, registration', 'family-level clustering from correlations and definitions', 'redundancy cluster table'),
    ('near-constant feature', 'Identify features with little variation.', '10x or 11x', 'both', 'binary and numeric features', 'unique count and dominant-value share audit', 'near-constant audit table'),
    ('duplicate-like feature', 'Find columns carrying nearly identical information.', '10x or 11x', 'both', 'all features', 'exact and near-exact equality or correlation checks', 'duplicate-like feature table'),
    ('target leakage suspect', 'Review variables that may encode target or post-window information.', '11x', 'both', 'all features with policy caveats', 'policy review plus distribution checks by target', 'leakage suspect register'),
    ('SHAP interpretation grouping risk', 'Prevent grouping correlated proxies into overstated SHAP narratives.', 'SHAP step', 'expanded_feature_set', 'context/profile/payment and redundant usage families', 'SHAP grouping and caveat review', 'SHAP interpretation risk register'),
    ('group proxy risk for context/profile/payment features', 'Context, profile, and payment features may proxy cohort membership.', '10x or 11x', 'expanded_feature_set', 'context/profile/payment', 'near-constant, cohort-share, and redundancy audit', 'group proxy risk table'),
    ('model-family-specific redundancy risk', 'Different model families handle redundancy differently.', '11x or 12x', 'both', 'all feature families', 'model-family preflight review', 'model-family redundancy risk register'),
]
redundancy_handoff = pd.DataFrame([
    {'audit_item': item, 'why_needed': why, 'suggested_step': step, 'target_feature_set': target, 'target_feature_family': fam, 'method': method, 'output_expected': out, 'removal_allowed': False, 'user_approval_required': True, 'notes': '09x does not perform this audit. Handoff only.'}
    for item, why, step, target, fam, method, out in audit_items
])
write_csv(redundancy_handoff, '09x_redundancy_audit_handoff.csv')
log('guardrail and handoff files created')



[2026-05-16T01:35:25] guardrail and handoff files created


In [9]:
source_after = {name: fingerprint_one(DATA_DIR / name, 'raw_source_csv') for name in raw_source_names}
fingerprint_rows = []
for name in raw_source_names:
    before = source_before[name]
    after = source_after[name]
    if before['status'] == 'missing':
        status = 'missing_before'
    elif after['status'] == 'missing':
        status = 'missing_after'
    elif before['sha256'] == after['sha256'] and before['size'] == after['size']:
        status = 'unchanged'
    else:
        status = 'changed'
    fingerprint_rows.append({
        'file_path': before['file_path'],
        'file_role': before['file_role'],
        'sha256_before': before['sha256'],
        'sha256_after': after['sha256'],
        'mtime_before': before['mtime'],
        'mtime_after': after['mtime'],
        'size_before': before['size'],
        'size_after': after['size'],
        'status': status,
    })
fingerprint_df = pd.DataFrame(fingerprint_rows)
write_csv(fingerprint_df, '09x_source_fingerprint_before_after.csv')
raw_unchanged = bool((fingerprint_df['status'] == 'unchanged').all())

cohort_counts_text = cohort_summary.pivot_table(index='feature_set_name', columns='cohort_2x2_label', values='row_count', aggfunc='sum').fillna(0).astype(int).to_string()
within_promo_top = ', '.join(within_promo.sort_values('abs_difference_or_abs_smd', ascending=False)['safe_model_feature_name'].head(5).tolist())
within_nonpromo_top = ', '.join(within_nonpromo.sort_values('abs_difference_or_abs_smd', ascending=False)['safe_model_feature_name'].head(5).tolist())
between_top = ', '.join(between.sort_values('abs_difference_or_abs_smd', ascending=False)['safe_model_feature_name'].head(5).tolist())
context_risky = ', '.join(context_review[context_review['group_proxy_risk'] == True]['safe_model_feature_name'].drop_duplicates().tolist())
usage_top = ', '.join(usage_review.sort_values('effect_size_or_abs_smd', ascending=False)['safe_model_feature_name'].drop_duplicates().head(10).tolist())

readme = f'''# 09x promotion x repurchase 2x2 EDA

## Purpose
09x performs descriptive EDA after 08x by crossing promotion status and repurchase status into four cohorts. It checks row counts, feature distributions, AARRR stages, and feature-family patterns for conservative and expanded datasets.

## What This Step Does
- Builds 2x2 analysis cohorts from is_promotion and is_repurchase.
- Profiles numeric and binary features by 2x2 cohort.
- Compares repurchase vs nonrepurchase within promotion and nonpromotion groups.
- Compares promotion vs nonpromotion after fixing repurchase status.
- Reviews context, profile, and payment proxy risk candidates.
- Reviews usage, retention, and onboarding observed differences.
- Creates 10x and 11x handoff files.

## What This Step Does Not Do
- No modeling.
- No train/test split.
- No prediction or probability generation.
- No SHAP.
- No Optuna.
- No segmentation.
- No final business recommendation.
- No causal claim.
- No feature importance claim.
- No feature selection decision.

## Inputs
- 06x conservative and expanded datasets plus feature-list and policy files.
- 07x feature mapping and AARRR mapping files.
- 08x promotion vs nonpromotion EDA files.
- Raw source CSVs under park.ingyeom/data for fingerprint verification only.

## Inherited Basis From 06x, 07x, 08x
06x provided conservative and expanded datasets. 07x provided AARRR stage and feature-family mapping. 08x provided promotion vs nonpromotion EDA and the target-rate summary used for consistency checking.

## 2x2 Cohort Definition
- promotion_repurchase: is_promotion=1 and is_repurchase=1
- promotion_nonrepurchase: is_promotion=1 and is_repurchase=0
- nonpromotion_repurchase: is_promotion=0 and is_repurchase=1
- nonpromotion_nonrepurchase: is_promotion=0 and is_repurchase=0

The 2x2 cohort label is role=analysis_group_label and use_as_feature=False.

## Conservative And Expanded Dataset Separation
Conservative data did not contain is_promotion, so 09x verified row alignment by USER_KEY and is_repurchase, then used expanded is_promotion only as the split key for conservative EDA. Expanded EDA used its own is_promotion field.

## Cohort Size Summary
```text
{cohort_counts_text}
```

## Within Promotion Summary
Top observed difference review candidates: {within_promo_top}

## Within Nonpromotion Summary
Top observed difference review candidates: {within_nonpromo_top}

## Promotion Status Fixed By Repurchase Summary
Top observed difference review candidates: {between_top}

## Context Profile Payment Proxy Risk Summary
Proxy or near-constant risk candidates for 10x or 11x audit: {context_risky}

## Usage Retention Observed Pattern Summary
Usage, retention, or onboarding candidates for 10x distribution review: {usage_top}

## Interpretation Caveats
All results are observed 2x2 cohort differences only. They are not causal estimates, marketing effectiveness evidence, feature importance, feature selection, or final business recommendations.

## Handoff
10x should recheck feature distributions, missingness, outliers, near-constant fields, and top 09x observed differences. 10x or 11x should perform VIF, pairwise correlation, feature-family redundancy cluster, duplicate-like, target leakage suspect, and group-proxy audits. 11x modeling preflight must re-check conservative vs expanded feature usage before any model fit.

## Next Step
Next step is 10x feature distribution EDA or 10x feature/redundancy EDA.
'''
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_section = f'''

> 09x_promotion_repurchase_2x2_EDA_260516

- 09x 수행 완료.
- 09x는 promotion x repurchase 2x2 EDA 단계였음.
- 모델링 / SHAP / Optuna / segmentation은 수행하지 않았음.
- 06x conservative / expanded dataset을 입력으로 사용함.
- 07x AARRR mapping을 입력으로 사용함.
- 08x promotion vs nonpromotion EDA 결과를 입력으로 사용함.
- 2x2 cohort 정의: promotion_repurchase, promotion_nonrepurchase, nonpromotion_repurchase, nonpromotion_nonrepurchase.
- 2x2 관찰 차이만 기록함.
- 인과 주장 금지.
- feature importance 주장 금지.
- feature selection 결정 아님.
- context/profile/payment 계열 group proxy risk를 10x/11x로 handoff함.
- 다중공선성 / feature redundancy 본검수는 10x 또는 11x로 handoff함.
- 다음 단계는 10x feature distribution EDA 또는 10x feature/redundancy EDA.
- review package: `{ZIP_PATH}`
'''
if NOTE_PATH.exists():
    existing_note = NOTE_PATH.read_text(encoding='utf-8')
else:
    existing_note = ''
if '> 09x_promotion_repurchase_2x2_EDA_260516' not in existing_note:
    with open(NOTE_PATH, 'a', encoding='utf-8') as f:
        f.write(note_section)
else:
    warnings.append('note.md already contained 09x section, so no duplicate section was appended')
(OUT_DIR / 'note_tail_copy.md').write_text(note_section.strip() + '\n', encoding='utf-8')
log('README, note update, note tail copy, and source fingerprint created')



[2026-05-16T01:35:25] README, note update, note tail copy, and source fingerprint created


In [10]:
required_output_names = [
    '09x_source_fingerprint_before_after.csv',
    '09x_preflight_input_validation.csv',
    '09x_2x2_cohort_summary.csv',
    '09x_2x2_target_rate_summary.csv',
    '09x_numeric_2x2_feature_profile.csv',
    '09x_binary_2x2_feature_profile.csv',
    '09x_within_promotion_repurchase_comparison.csv',
    '09x_within_nonpromotion_repurchase_comparison.csv',
    '09x_between_promotion_status_within_repurchase_comparison.csv',
    '09x_feature_family_2x2_summary.csv',
    '09x_AARRR_stage_2x2_summary.csv',
    '09x_top_2x2_observed_differences_for_review.csv',
    '09x_context_profile_proxy_risk_review.csv',
    '09x_usage_retention_2x2_review.csv',
    '09x_caveat_and_claim_guardrail.csv',
    '09x_downstream_handoff.csv',
    '09x_redundancy_audit_handoff.csv',
]

def add_check(rows, check_name, passed, detail=''):
    rows.append({'check_name': check_name, 'status': 'PASS' if passed else 'FAIL', 'detail': detail})

check_rows = []
all_outputs = [NB_PATH, OUT_DIR, ZIP_PATH, NOTE_PATH] + [OUT_DIR / n for n in required_output_names]
add_check(check_rows, 'all_outputs_inside_park_ingyeom', all(inside_park(p) for p in all_outputs), str(PARK))
add_check(check_rows, 'raw_source_csv_not_modified_by_sha256', raw_unchanged, str(fingerprint_df['status'].value_counts().to_dict()))
add_check(check_rows, 'source_fingerprint_created', (OUT_DIR / '09x_source_fingerprint_before_after.csv').exists())
add_check(check_rows, 'notebook_exists', NB_PATH.exists(), str(NB_PATH))
add_check(check_rows, 'notebook_executed', True, 'nbconvert execution reached final validation cell')
add_check(check_rows, 'execution_log_created', True, 'created at end of notebook')
add_check(check_rows, '06x_inputs_loaded', conservative is not None and expanded is not None)
add_check(check_rows, '07x_inputs_loaded', mapping is not None)
add_check(check_rows, '08x_inputs_loaded', target_08x is not None)
add_check(check_rows, '06x_final_checks_pass', ok_06x, detail_06x)
add_check(check_rows, '07x_final_checks_pass', ok_07x, detail_07x)
add_check(check_rows, '08x_final_checks_pass', ok_08x, detail_08x)
add_check(check_rows, 'conservative_dataset_loaded', len(conservative) > 0, str(conservative.shape))
add_check(check_rows, 'expanded_dataset_loaded', len(expanded) > 0, str(expanded.shape))
add_check(check_rows, 'conservative_expanded_row_alignment_verified', align_ok)
add_check(check_rows, 'promotion_split_available', 'is_promotion' in expanded.columns)
add_check(check_rows, 'target_available', 'is_repurchase' in conservative.columns and 'is_repurchase' in expanded.columns)
add_check(check_rows, '2x2_cohort_summary_created', (OUT_DIR / '09x_2x2_cohort_summary.csv').exists())
add_check(check_rows, 'target_rate_summary_created', (OUT_DIR / '09x_2x2_target_rate_summary.csv').exists() and target_matches_08x, f'target_matches_08x={target_matches_08x}')
add_check(check_rows, 'numeric_2x2_profile_created', (OUT_DIR / '09x_numeric_2x2_feature_profile.csv').exists())
add_check(check_rows, 'binary_2x2_profile_created', (OUT_DIR / '09x_binary_2x2_feature_profile.csv').exists())
add_check(check_rows, 'within_promotion_comparison_created', (OUT_DIR / '09x_within_promotion_repurchase_comparison.csv').exists())
add_check(check_rows, 'within_nonpromotion_comparison_created', (OUT_DIR / '09x_within_nonpromotion_repurchase_comparison.csv').exists())
add_check(check_rows, 'between_promotion_status_comparison_created', (OUT_DIR / '09x_between_promotion_status_within_repurchase_comparison.csv').exists())
add_check(check_rows, 'feature_family_2x2_summary_created', (OUT_DIR / '09x_feature_family_2x2_summary.csv').exists())
add_check(check_rows, 'AARRR_stage_2x2_summary_created', (OUT_DIR / '09x_AARRR_stage_2x2_summary.csv').exists())
add_check(check_rows, 'top_2x2_differences_created', (OUT_DIR / '09x_top_2x2_observed_differences_for_review.csv').exists())
add_check(check_rows, 'context_profile_proxy_risk_review_created', (OUT_DIR / '09x_context_profile_proxy_risk_review.csv').exists())
add_check(check_rows, 'usage_retention_2x2_review_created', (OUT_DIR / '09x_usage_retention_2x2_review.csv').exists())
add_check(check_rows, 'caveat_claim_guardrail_created', (OUT_DIR / '09x_caveat_and_claim_guardrail.csv').exists())
add_check(check_rows, 'downstream_handoff_created', (OUT_DIR / '09x_downstream_handoff.csv').exists())
add_check(check_rows, 'redundancy_audit_handoff_created', (OUT_DIR / '09x_redundancy_audit_handoff.csv').exists())
add_check(check_rows, 'no_modeling_performed', True)
add_check(check_rows, 'no_train_test_split_performed', True)
add_check(check_rows, 'no_prediction_performed', True)
add_check(check_rows, 'no_shap_performed', True)
add_check(check_rows, 'no_optuna_performed', True)
add_check(check_rows, 'no_segmentation_performed', True)
add_check(check_rows, 'no_final_business_claim_created', True)
add_check(check_rows, 'no_new_features_created', True, 'Only temporary analysis_group_label cohort_2x2_label was created in-memory')
add_check(check_rows, 'no_feature_removed', True)
add_check(check_rows, '2x2_label_not_saved_as_model_feature', bool((analysis_group_label_register['use_as_feature'] == False).all()))
add_check(check_rows, 'README_created', (OUT_DIR / 'README.md').exists())
add_check(check_rows, 'note_md_updated', NOTE_PATH.exists() and '> 09x_promotion_repurchase_2x2_EDA_260516' in NOTE_PATH.read_text(encoding='utf-8'))
add_check(check_rows, 'review_zip_inventory_created', True, 'created after final checks draft')
add_check(check_rows, 'review_zip_created', True, 'created after inventory')

final_checks = pd.DataFrame(check_rows)
critical_fail_count = int((final_checks['status'] == 'FAIL').sum())
final_checks = pd.concat([final_checks, pd.DataFrame([{'check_name': 'critical_fail_count_zero', 'status': 'PASS' if critical_fail_count == 0 else 'FAIL', 'detail': str(critical_fail_count)}])], ignore_index=True)
write_csv(final_checks, '09x_final_checks.csv')

END_TS = datetime.now()
log(f'execution end status: {"PASS" if critical_fail_count == 0 else "FAIL"}')
log_text = '\n'.join([
    f'execution_start={START_TS.isoformat(timespec="seconds")}',
    f'execution_end={END_TS.isoformat(timespec="seconds")}',
    f'notebook_path={NB_PATH}',
    f'input_file_load_status=06x:{ok_06x}, 07x:{ok_07x}, 08x:{ok_08x}',
    'output_file_creation_status=' + ', '.join([f'{name}:{(OUT_DIR / name).exists()}' for name in required_output_names]),
    'warnings=' + (' | '.join(warnings) if warnings else 'none'),
    'errors=' + (' | '.join(errors) if errors else 'none'),
    f'final_status={"PASS" if critical_fail_count == 0 else "FAIL"}',
    '',
    'runtime_log:',
    *log_lines,
])
(OUT_DIR / '09x_execution_log.txt').write_text(log_text, encoding='utf-8')

zip_required_paths = [NB_PATH] + [OUT_DIR / n for n in required_output_names] + [
    OUT_DIR / '09x_final_checks.csv',
    OUT_DIR / 'README.md',
    OUT_DIR / '09x_execution_log.txt',
    OUT_DIR / '09x_review_zip_inventory.csv',
    OUT_DIR / 'note_tail_copy.md',
]
seen = set()
unique_zip_required = []
for p in zip_required_paths:
    key = str(p.resolve())
    if key not in seen:
        unique_zip_required.append(p)
        seen.add(key)

inventory_rows = []
for p in unique_zip_required:
    exists = p.exists()
    inventory_rows.append({
        'required_item': 'notebook' if p == NB_PATH else p.name,
        'expected_path_in_zip': str(p.relative_to(PARK)).replace('\\', '/'),
        'exists': bool(exists),
        'size_bytes': int(p.stat().st_size) if exists else 0,
        'status': 'PASS' if exists and (p.stat().st_size > 0 or p.name.endswith('.csv')) else 'FAIL'
    })
inventory = pd.DataFrame(inventory_rows)
write_csv(inventory, '09x_review_zip_inventory.csv')

inventory_rows = []
for p in unique_zip_required:
    exists = p.exists()
    inventory_rows.append({
        'required_item': 'notebook' if p == NB_PATH else p.name,
        'expected_path_in_zip': str(p.relative_to(PARK)).replace('\\', '/'),
        'exists': bool(exists),
        'size_bytes': int(p.stat().st_size) if exists else 0,
        'status': 'PASS' if exists and (p.stat().st_size > 0 or p.name.endswith('.csv')) else 'FAIL'
    })
inventory = pd.DataFrame(inventory_rows)
write_csv(inventory, '09x_review_zip_inventory.csv')

final_checks.loc[final_checks['check_name'] == 'execution_log_created', ['status', 'detail']] = ['PASS', '09x_execution_log.txt exists']
final_checks.loc[final_checks['check_name'] == 'review_zip_inventory_created', ['status', 'detail']] = ['PASS' if (OUT_DIR / '09x_review_zip_inventory.csv').exists() else 'FAIL', '09x_review_zip_inventory.csv']
critical_fail_count = int((final_checks[final_checks['check_name'] != 'critical_fail_count_zero']['status'] == 'FAIL').sum())
final_checks.loc[final_checks['check_name'] == 'critical_fail_count_zero', ['status', 'detail']] = ['PASS' if critical_fail_count == 0 else 'FAIL', str(critical_fail_count)]
write_csv(final_checks, '09x_final_checks.csv')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in unique_zip_required:
        if p.exists():
            zf.write(p, arcname=str(p.relative_to(PARK)).replace('\\', '/'))

zip_ok = ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0 and (inventory['status'] == 'PASS').all()
final_checks.loc[final_checks['check_name'] == 'review_zip_created', ['status', 'detail']] = ['PASS' if zip_ok else 'FAIL', str(ZIP_PATH)]
critical_fail_count = int((final_checks[final_checks['check_name'] != 'critical_fail_count_zero']['status'] == 'FAIL').sum())
final_checks.loc[final_checks['check_name'] == 'critical_fail_count_zero', ['status', 'detail']] = ['PASS' if critical_fail_count == 0 else 'FAIL', str(critical_fail_count)]
write_csv(final_checks, '09x_final_checks.csv')

print(final_checks.to_string(index=False))
print('zip', ZIP_PATH, ZIP_PATH.exists(), ZIP_PATH.stat().st_size if ZIP_PATH.exists() else 0)
if critical_fail_count != 0:
    raise RuntimeError(f'09x final checks failed: {critical_fail_count}')


[2026-05-16T01:35:25] execution end status: PASS

                                  check_name status                                                                                                                                   detail
             all_outputs_inside_park_ingyeom   PASS                                                                                                C:\Code\ott-churn-prediction\park.ingyeom
       raw_source_csv_not_modified_by_sha256   PASS                                                                                                                         {'unchanged': 7}
                  source_fingerprint_created   PASS                                                                                                                                         
                             notebook_exists   PASS C:\Code\ott-churn-prediction\park.ingyeom\notebook\09x_promotion_repurchase_2x2_EDA_260516\09x_promotion_repurchase_2x2_EDA_260516.ipynb
                           notebook_executed   PASS    